🧠 Phase 1 — Install & Load via OpenML API

In [ ]:
!pip install openml

In [1]:
import openml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Download dataset from OpenML
dataset = openml.datasets.get_dataset(40927)

X, y, _, _ = dataset.get_data(target=dataset.default_target_attribute)

print(X.shape)
print(y.shape)

(60000, 3072)
(60000,)


🧱 Phase 2 — Reshape for CNN

In [2]:
X = X.to_numpy().astype("float32") / 255.0
y = y.astype("int")

# Reshape properly
X = X.reshape(-1, 3, 32, 32)
X = np.transpose(X, (0, 2, 3, 1))  # (N, 32, 32, 3)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape)

(48000, 32, 32, 3)


🏗 Phase 3 — Baseline CNN Model(For RGB)

In [17]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, 3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 30, 30, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 13, 13, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 4, 4, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 228,042 (890.79 KB)

 Trainable params: 227,594 (889.04 KB)

 Non-trainable params: 448 (1.75 KB)

🚀 Phase 3 — Train Baseline Model

In [18]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val, y_val)
)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.3762 - loss: 1.8495 - val_accuracy: 0.5175 - val_loss: 1.3120
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5854 - loss: 1.1713 - val_accuracy: 0.6127 - val_loss: 1.0781
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.6585 - loss: 0.9749 - val_accuracy: 0.6612 - val_loss: 0.9649
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7054 - loss: 0.8502 - val_accuracy: 0.6023 - val_loss: 1.1455
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7358 - loss: 0.7657 - val_accuracy: 0.6401 - val_loss: 1.0718
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7599 - loss: 0.6949 - val_accuracy: 0.7098 - val_loss: 0.8886
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7814 - loss: 0.6301 - val_accuracy: 0.7122 - val_loss: 0.8837
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7976 - loss: 0.5782 - val_accuracy: 0

Expected:

~65–75% validation accuracy

If we hit 75%+, we built a solid baseline.

🚀 Phase 4 Transfer Learning (Now It Gets Serious)

Resize to 96×96:
For Mobilnet Model

In [4]:
import tensorflow as tf
X_train_resized = tf.image.resize(X_train, (96, 96))
X_val_resized = tf.image.resize(X_val, (96, 96))

Load MobileNet:

In [5]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96,96,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

Add head:

In [7]:
from tensorflow.keras import layers, models
inputs = tf.keras.Input(shape=(96,96,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model_mobilenet = tf.keras.Model(inputs, outputs)

model_mobilenet.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train (Frozen Backbone Phase)

This is the first stage — feature extraction only.

In [8]:
history_mobilenet = model_mobilenet.fit(
    X_train_resized, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val_resized, y_val),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=2,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 49s 37ms/step - accuracy: 0.5954 - loss: 1.1973 - val_accuracy: 0.7679 - val_loss: 0.6528 - learning_rate: 0.0010
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.7430 - loss: 0.7450 - val_accuracy: 0.7929 - val_loss: 0.5857 - learning_rate: 0.0010
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.7648 - loss: 0.6817 - val_accuracy: 0.7897 - val_loss: 0.5922 - learning_rate: 0.0010
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.7804 - loss: 0.6406 - val_accuracy: 0.8013 - val_loss: 0.5624 - learning_rate: 0.0010
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.7861 - loss: 0.6209 - val_accuracy: 0.7985 - val_loss: 0.5585 - learning_rate: 0.0010
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.7965 - loss: 0.5858 - val_accuracy: 0.8032 - val_loss: 0.5575 - learning_rate: 0.0010
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.7995 - l

Expected:

~80–85%

🎯 Then Fine-Tune

Unfreeze last 60 layers:

In [9]:
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Now train again:

In [10]:
history_finetune = model_mobilenet.fit(
    X_train_resized, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val_resized, y_val),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=2,
            restore_best_weights=True
        )
    ],
    verbose=1
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 51s 38ms/step - accuracy: 0.6342 - loss: 1.6364 - val_accuracy: 0.8173 - val_loss: 0.5983
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 21s 29ms/step - accuracy: 0.7735 - loss: 0.7922 - val_accuracy: 0.8372 - val_loss: 0.5108
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 22s 29ms/step - accuracy: 0.8069 - loss: 0.6154 - val_accuracy: 0.8465 - val_loss: 0.4766
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.8291 - loss: 0.5243 - val_accuracy: 0.8532 - val_loss: 0.4596
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 35s 28ms/step - accuracy: 0.8498 - loss: 0.4558 - val_accuracy: 0.8592 - val_loss: 0.4404


In [ ]:
import sys
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn
import openml
import cv2
import h5py

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", seaborn.__version__)
print("OpenML:", openml.__version__)
print("OpenCV:", cv2.__version__)
print("h5py:", h5py.__version__)

In [13]:
# Save best model
model_mobilenet.save("mobilenet_cifar10_best.keras")